In [1]:
import boto3
import math
from sagemaker import get_execution_role
from pprint import pprint
import time

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sagemaker/base_serializers.py:28: UserWarning: A NumPy version >=1.23.5 and <2.3.0 is required for this version of SciPy (detected version 1.22.4)
  import scipy.sparse


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


### Functions

In [2]:
def get_specs(str_instance):
    if str_instance == 'm5.large':
        int_vcpu = 2
        int_memory_gb = 8
    elif str_instance == 'm5.xlarge':
        int_vcpu = 4
        int_memory_gb = 16
    elif str_instance == 'm5.2xlarge':
        int_vcpu = 8
        int_memory_gb = 32
    elif str_instance == 'm5.4xlarge':
        int_vcpu = 16
        int_memory_gb = 64
    elif str_instance == 'm5.8xlarge':
        int_vcpu = 32
        int_memory_gb = 128
    elif str_instance == 'm5.12xlarge':
        int_vcpu = 48
        int_memory_gb = 192
    int_memory_mebibytes = math.ceil(int_memory_gb * 953.674)
    dict_output = {
        'int_vcpu': int_vcpu,
        'int_memory_gb': int_memory_gb,
        'int_memory_mebibytes': int_memory_mebibytes,
    }
    return dict_output

### Constants

In [3]:
str_image_name = 'genxii-early-30-360'
int_iteration = 1
str_instance = 'm5.4xlarge'
dict_specs = get_specs(str_instance=str_instance)
int_vcpu = dict_specs['int_vcpu']
int_memory_gb = dict_specs['int_memory_gb']
int_memory_mebibytes = dict_specs['int_memory_mebibytes']
for key, val in dict_specs.items():
    print(f'{key}: {val}')

int_vcpu: 16
int_memory_gb: 64
int_memory_mebibytes: 61036


### Create compute environment

In [4]:
# initialize class
cls_client = boto3.client('batch')

In [5]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [6]:
# create compute environment
while True:
    try:
        str_compute_env_name = f'env-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_compute_environment(
            computeEnvironmentName=str_compute_env_name,
            type= 'Managed', 
            state= 'ENABLED',
            serviceRole = str_role,
            computeResources={
                #'type': 'SPOT',
                'type': 'EC2',
                'minvCpus': 0,
                'maxvCpus': 256, 
                'desiredvCpus': int_vcpu,
                'instanceTypes': [
                    str_instance,
                ], 
                'subnets': ['subnet-044e573651bb251a7'], 
                'securityGroupIds': ['sg-03904237048cdc335'], 
                'instanceRole': 'ecsInstanceRole',
                #'spotIamFleetRole': 'AmazonEC2SpotFleetTaggingRole',
            },
        )
        pprint(dict_response)
        break
    except:
        int_iteration += 1

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '163',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 29 Aug 2024 19:41:43 GMT',
                                      'x-amz-apigw-id': 'dSVGnG0sPHcEZ3Q=',
                                      'x-amzn-requestid': 'bd5f9859-ad95-4059-a4c6-f40cb145bd40',
                                      'x-amzn-trace-id': 'Root=1-66d0cef6-51b36b0b2cba7d7c4b45add6'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'bd5f9859-ad95-4059-a4c6-f40cb145bd40',
                      'RetryAttempts': 0},
 'computeEnvironmentArn': 'arn:aws:batch:

### Create Job Queue

In [7]:
# create job queue (this is where AWS will store your jobs until an EC2 Instance is available to run them)
while True:
    try:
        str_job_queue_name = f'queue-{str_image_name}-{int_iteration}'
        dict_response = cls_client.create_job_queue(
            jobQueueName=str_job_queue_name,
            state='ENABLED',
            priority=1,
            computeEnvironmentOrder=[
                {
                    'order': 1,
                    'computeEnvironment': str_compute_env_name,
                },
            ]
        )
        # get arn
        str_job_queue_arn = dict_response['jobQueueArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '137',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 29 Aug 2024 19:42:05 GMT',
                                      'x-amz-apigw-id': 'dSVKKFH2PHcEeOQ=',
                                      'x-amzn-requestid': 'b7ed75c2-b136-445f-9713-d275e1a8e19a',
                                      'x-amzn-trace-id': 'Root=1-66d0cf0d-439ca90e2a7c79153aaf4fbf'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'b7ed75c2-b136-445f-9713-d275e1a8e19a',
                      'RetryAttempts': 0},
 'jobQueueArn': 'arn:aws:batch:us-west-2:

### Register job definition

In [8]:
# job definition
while True:
    try:
        str_job_definition = f'job-def-{str_image_name}-{int_iteration}'
        dict_response = cls_client.register_job_definition(
            type='container',
            containerProperties={
                'image': f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_image_name}:latest',
                'memory': int_memory_mebibytes,
                'vcpus': int_vcpu,
            },
            jobDefinitionName=str_job_definition,
        )
        # get arn
        str_job_def_arn = dict_response['jobDefinitionArn']
        pprint(dict_response)
        break
    except:
        time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '171',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 29 Aug 2024 19:42:05 GMT',
                                      'x-amz-apigw-id': 'dSVKLFUgPHcEjhw=',
                                      'x-amzn-requestid': 'a4827db4-9f20-4578-9004-0335acb01b2f',
                                      'x-amzn-trace-id': 'Root=1-66d0cf0d-0a59ce184123738510fe362b'},
                      'HTTPStatusCode': 200,
                      'RequestId': 'a4827db4-9f20-4578-9004-0335acb01b2f',
                      'RetryAttempts': 0},
 'jobDefinitionArn': 'arn:aws:batch:us-we

### Submit job

In [9]:
# # submit a job (only for testing)
# while True:
#     try:
#         str_job_name = f'job-name-{str_image_name}-{int_iteration}'
#         response = cls_client.submit_job(
#             jobDefinition=str_job_definition,
#             jobQueue=str_job_queue_name,
#             jobName=str_job_name,
#         )
#         pprint(response)
#         break
#     except:
#         time.sleep(1)

{'ResponseMetadata': {'HTTPHeaders': {'access-control-allow-origin': '*',
                                      'access-control-expose-headers': 'X-amzn-errortype,X-amzn-requestid,X-amzn-errormessage,X-amzn-trace-id,X-amz-apigw-id,date',
                                      'connection': 'keep-alive',
                                      'content-length': '180',
                                      'content-type': 'application/json',
                                      'date': 'Thu, 29 Aug 2024 19:42:08 GMT',
                                      'x-amz-apigw-id': 'dSVKiGvRvHcEldA=',
                                      'x-amzn-requestid': '4df48631-bd99-487f-8cc0-6af6e177c880',
                                      'x-amzn-trace-id': 'Root=1-66d0cf0f-07d6c89a2966ff6651c392a7'},
                      'HTTPStatusCode': 200,
                      'RequestId': '4df48631-bd99-487f-8cc0-6af6e177c880',
                      'RetryAttempts': 0},
 'jobArn': 'arn:aws:batch:us-west-2:83669

### Show arns

In [10]:
print(f'Job Queue ARN: {str_job_queue_arn}')
print(f'Job Definition ARN: {str_job_def_arn}')

Job Queue ARN: arn:aws:batch:us-west-2:836690756591:job-queue/queue-genxii-early-30-360-1
Job Definition ARN: arn:aws:batch:us-west-2:836690756591:job-definition/job-def-genxii-early-30-360-1:1
